In [1]:
import numpy as np
import os
import pandas as pd
from matplotlib import pyplot as plt

In [2]:
def make_canvas(
    axes_width_pt: float = 300.0,
    axes_aspect: float = 2/3,
    left_pt: float = 40.0,
    right_pt: float = 20.0,
    bottom_pt: float = 35.0,
    top_pt: float = 20.0,
    fontsize: float = 8.0,
):
    _PT_PER_IN = 72.0
    # Use PDF “base 14” fonts (Helvetica) — no TTF embedding, no fontTools warnings
    plt.rcParams.update({
        "pdf.use14corefonts": True,   # key line
        "ps.useafm": True,            # for .ps if you ever use it
        # Do NOT set pdf.fonttype/ps.fonttype when using core fonts
        "text.usetex": False,         # set True only if you want LaTeX (see Option C)
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica"],
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize,
        "ytick.labelsize": fontsize,
        "legend.fontsize": fontsize,
    })
    # Make math text look sans-ish to match Helvetica
    plt.rcParams.update({
        "mathtext.fontset": "stixsans",
    })

    axes_h_pt = axes_width_pt * float(axes_aspect)
    fig_w_pt = left_pt + axes_width_pt + right_pt
    fig_h_pt = bottom_pt + axes_h_pt + top_pt

    fig = plt.figure(figsize=(fig_w_pt/_PT_PER_IN, fig_h_pt/_PT_PER_IN))
    ax = fig.add_axes([
        left_pt/fig_w_pt,
        bottom_pt/fig_h_pt,
        axes_width_pt/fig_w_pt,
        axes_h_pt/fig_h_pt,
    ])
    # ax.grid(True, which="both", linestyle=":", linewidth=0.5)
    return fig, ax

## Ploylog plot

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_error_vs_d(csv_path, canvas_kwargs=None, savepath=None, show=True):
    """
    Read error data from csv_path and plot mean error vs d (log-log),
    averaging over seeds t and using one standard deviation as error bar.

    Assumes columns: m, k, d, t, least_error.
    Only d, t, least_error are actually used here.
    """
    if canvas_kwargs is None:
        canvas_kwargs = {}

    # Load data
    df = pd.read_csv(csv_path)

    # Group by d and average over seeds (t)
    grouped = df.groupby("d")["least_error"]
    mean_err = grouped.mean()
    std_err = grouped.std()

    # Sort by d (groupby index is d)
    d_vals = mean_err.index.to_numpy()
    mean_vals = mean_err.to_numpy()
    std_vals = std_err.to_numpy()

    # Make canvas (your own function)
    fig, ax = make_canvas(**canvas_kwargs)

    # Error bar plot in log-log scale
    ax.errorbar(
        d_vals,
        mean_vals,
        yerr=std_vals,
        fmt="o-",
        capsize=3,
        linewidth=1.5,
        markersize=4,
        label="mean ± 1 std",
    )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel(r"$d$")
    ax.set_ylabel(r"$\mathrm{error}$")
    ax.legend()

    fig.tight_layout()

    if savepath is not None:
        fig.savefig(savepath, bbox_inches="tight")

    if show:
        plt.show()

    return fig, ax

In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_error_vs_d(csv_path, savepath=None, show=True, **canvas_kwargs):
    """
    Read error data from csv_path and plot mean error vs d (log-log),
    averaging over seeds t and using one standard deviation as error bar.

    Assumes columns: m, k, d, t, least_error.
    Only d, t, least_error are actually used here.
    """
    if canvas_kwargs is None:
        canvas_kwargs = {}

    # Load data
    df = pd.read_csv(csv_path)

    # Group by d and average over seeds (t)
    grouped = df.groupby("d")["least_error"]
    mean_err = grouped.mean()
    std_err = grouped.std()

    # Sort by d (groupby index is d)
    d_vals = mean_err.index.to_numpy()
    mean_vals = mean_err.to_numpy()
    std_vals = std_err.to_numpy()

    # Make canvas (your own function)
    fig, ax = make_canvas(**canvas_kwargs)

    # Error bar plot in log-log scale
    ax.errorbar(
        d_vals[1:],
        mean_vals[1:],
        yerr=std_vals[1:],
        fmt="o-",
        capsize=3,
        linewidth=1.5,
        markersize=4,
        label="mean ± 1 std",
    )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim([1000, 80000])

    ax.set_xlabel(r"$d$")
    ax.set_ylabel(r"$\left|f(\theta) - f(\theta')\right|$")
    ax.set_title(r"$m=1, d\to [120\,\log d]$")
    # ax.set_title(r"$m=2, d\to [20\,\log^2 d]$")
    # ax.legend()

    fig.tight_layout()

    if savepath is not None:
        fig.savefig(savepath, bbox_inches="tight")

    if show:
        plt.show()

    return fig, ax

In [ ]:
fig, ax = plot_error_vs_d(
    "polylog_error_1d.csv",
    savepath="polylog_m1.pdf",
    axes_width_pt=150.
)